# 🌊❄️ Prefect + dbt + Snowflake — Intro & Setup

---

## 🏗️ Architecture

```
Prefect Flow
    │
    ├── @task: dbt seed   → Load CSVs into Snowflake RAW schema
    ├── @task: dbt run    → Transform via staging → marts models
    ├── @task: dbt test   → Data quality checks on Snowflake tables
    └── on_failure hook   → Send alert if anything breaks
                                    │
                               Snowflake
                           (Warehouse / DB / Schema)
```

---

## 💻 Example 1: Install Required Packages

In [ ]:
# Run these in terminal (not in notebook to avoid kernel conflicts)
install_commands = [
    "pip install prefect",
    "pip install dbt-snowflake",      # dbt core + Snowflake adapter
    "pip install prefect-dbt",        # Optional: official Prefect-dbt integration
    "pip install snowflake-connector-python",
]
for cmd in install_commands:
    print(f"  $ {cmd}")

# Verify prefect
import warnings; warnings.filterwarnings('ignore')
import prefect
print(f"\n✅ Prefect {prefect.__version__} ready")

try:
    import dbt.version
    print(f"✅ dbt {dbt.version.__version__} ready")
except ImportError:
    print("❌ Run: pip install dbt-snowflake")

---

## 💻 Example 2: Snowflake profiles.yml Setup

dbt connects to Snowflake via a **`profiles.yml`** file.  
This file lives inside your dbt project folder (or `~/.dbt/profiles.yml`).

In [ ]:
# What your profiles.yml should look like for Snowflake:
profiles_yml_content = """
my_etl_project:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: YOUR_ACCOUNT.snowflakecomputing.com   # e.g. abc12345.us-east-1
      user: YOUR_USERNAME
      password: YOUR_PASSWORD
      role: TRANSFORMER                              # Snowflake role
      warehouse: COMPUTE_WH                         # Snowflake warehouse name
      database: ANALYTICS                           # Target database
      schema: dbt_dev                               # Dev schema
      threads: 4
      client_session_keep_alive: false

    prod:
      type: snowflake
      account: YOUR_ACCOUNT.snowflakecomputing.com
      user: YOUR_USERNAME
      password: "{{ env_var('SNOWFLAKE_PASSWORD') }}"  # ← Use env var in prod!
      role: TRANSFORMER
      warehouse: COMPUTE_WH
      database: ANALYTICS
      schema: dbt_prod
      threads: 8
"""
print(profiles_yml_content)
print("💡 Tip: NEVER hardcode passwords in profiles.yml for production. Use env_var() instead.")

---

## 💻 Example 3: Storing Snowflake Credentials in Prefect Secrets

In [ ]:
from prefect.blocks.system import Secret

# Run this ONCE to save credentials securely in Prefect Cloud
# After this, your flows can retrieve them without hardcoding

save_secrets_code = """
# Run this once to store credentials
Secret(value="YOUR_SNOWFLAKE_ACCOUNT").save("snowflake-account")
Secret(value="YOUR_SNOWFLAKE_USER").save("snowflake-user")
Secret(value="YOUR_SNOWFLAKE_PASSWORD").save("snowflake-password")
Secret(value="COMPUTE_WH").save("snowflake-warehouse")
Secret(value="ANALYTICS").save("snowflake-database")
"""
print("Run this once to store Snowflake credentials:")
print(save_secrets_code)

# Then retrieve like this in your flows:
retrieve_code = """
# In your flow/task:
sf_password = Secret.load("snowflake-password").get()
sf_account  = Secret.load("snowflake-account").get()
"""
print("To retrieve secrets in your flow:")
print(retrieve_code)

---

## 💻 Example 4: Verify dbt Can Connect to Snowflake

In [ ]:
import subprocess
import os

DBT_PROJECT_DIR = "/Users/aviraljain/Downloads/python advanced/my_etl_project"

# Optional: inject password from environment variable at runtime
# os.environ["SNOWFLAKE_PASSWORD"] = Secret.load("snowflake-password").get()

result = subprocess.run(
    ["dbt", "debug",
     "--project-dir", DBT_PROJECT_DIR,
     "--profiles-dir", DBT_PROJECT_DIR],
    capture_output=True, text=True
)

# Print last few lines (connection status)
for line in result.stdout.strip().split('\n')[-8:]:
    print(line)

if result.returncode == 0:
    print("\n✅ dbt can connect to Snowflake!")
else:
    print("\n❌ Connection failed — check your profiles.yml credentials")

---

## 🏭 Summary

| Config Item | Where It Goes | Example Value |
|---|---|---|
| `type: snowflake` | `profiles.yml` | Tells dbt to use Snowflake adapter |
| `account` | `profiles.yml` | `abc123.us-east-1` |
| `warehouse` | `profiles.yml` | `COMPUTE_WH` |
| `database` | `profiles.yml` | `ANALYTICS` |
| `schema` | `profiles.yml` | `dbt_dev` or `dbt_prod` |
| Password | `Prefect Secret` | Never hardcode — use `env_var()` |

---

## ⚠️ Common Beginners' Mistakes

In [ ]:
mistakes = [
    ("Hardcoding password in profiles.yml",    "Use env_var('SNOWFLAKE_PASSWORD') or Prefect Secrets"),
    ("Wrong account format",                    "Use full format: abc12345.us-east-1 (not just abc12345)"),
    ("No warehouse set",                        "Snowflake needs an active warehouse to run queries"),
    ("Using the ACCOUNTADMIN role in dbt",      "Create a dedicated TRANSFORMER role with least-privilege access"),
]
for mistake, fix in mistakes:
    print(f"❌ {mistake}")
    print(f"✅ Fix: {fix}\n")